In [ ]:
#THIS IS A LEGACY CODE

system = HeisenbergJ1J2(SquareLattice(width=4, height=4), J1=1, J2=0, use_symmetries=False, spin_inversion=None)
system.get_ground_state()
df = system.get_df_ground_state(expand_basis_columns=True)

df = df.sort_values('amplitude').reset_index(drop=True)[len(df) // 10:].assign(sign=lambda x: np.sign(x['ground_state']))

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score

df

df.columns[df.columns.str.match(r"s\d")]

knn = KNeighborsClassifier(n_neighbors=2, metric='hamming')
loo = LeaveOneOut()
X = df[df.columns[df.columns.str.match(r"s\d")]]
y = df['sign']
cross_val_score(knn, X, y, cv=loo, scoring='accuracy').mean()

knn = KNeighborsClassifier(n_neighbors=1, metric='hamming')
loo = LeaveOneOut()
X = df[df.columns[df.columns.str.match(r"s\d")]]
y = df['sign']
cross_val_score(knn, X, y, cv=loo, scoring='accuracy').mean()

system = HeisenbergJ1J2(SquareLattice(width=4, height=4), J1=1, J2=1, use_symmetries=False, spin_inversion=None)
system.get_ground_state()
df = system.get_df_ground_state(expand_basis_columns=True)

df = df.sort_values('amplitude').reset_index(drop=True)[len(df) // 10:].assign(sign=lambda x: np.sign(x['ground_state']))

knn = KNeighborsClassifier(n_neighbors=2, metric='hamming')
loo = LeaveOneOut()
X = df[df.columns[df.columns.str.match(r"s\d")]]
y = df['sign']
cross_val_score(knn, X, y, cv=loo, scoring='accuracy').mean()

cross_val_score(knn, X, y, cv=loo, scoring='accuracy').mean()

from tqdm.auto import tqdm
import jsonlines

loo = LeaveOneOut()

results = []
with jsonlines.open('square-lattice.jsonl', mode='w') as writer:
    for J2 in tqdm(np.linspace(0, 1, 11)):
        system = HeisenbergJ1J2(SquareLattice(width=4, height=4), J1=1, J2=J2, 
                                use_symmetries=False, 
                                spin_inversion=None)
        system.get_ground_state()
        df = system.get_df_ground_state(expand_basis_columns=True)
        df = df.sort_values('amplitude').reset_index(drop=True)[len(df) // 10:].assign(sign=lambda x: np.sign(x['ground_state']))
        X = df[df.columns[df.columns.str.match(r"s\d")]]
        y = df['sign']
        sign_counts = df['sign'].value_counts()
        disbalance = sign_counts[1.] / sign_counts[-1.]
        for k in [1, 2]:
            knn = KNeighborsClassifier(n_neighbors=k, metric='hamming')
            score = cross_val_score(knn, X, y, cv=loo, scoring='accuracy').mean()
            result = dict(J2=J2, k=k, score=score, disbalance=disbalance)
            print(result)
            result['df'] = df.to_json()
            results.append(result)
            writer.write(result)

res_df = pd.DataFrame(results)

res_df.drop('df', axis=1).assign(acc_plus=lambda x: x['disbalance'] / (x['disbalance'] + 1),
                                 acc_minus=lambda x: 1 / (x['disbalance'] + 1)).melt('J2', ['score', 'acc_plus', 'acc_minus'])

sns.lineplot(data=
             res_df.drop('df', axis=1).assign(acc_plus=lambda x: x['disbalance'] / (x['disbalance'] + 1),
                                 acc_minus=lambda x: 1 / (x['disbalance'] + 1)),
             x='J2', y='score')

sns.lineplot(data=
res_df.drop('df', axis=1)[lambda x: x['k'] == 2].assign(score=lambda x: np.where(x['score'] > 1/2, x['score'], 1 - x['score']), acc_plus=lambda x: x['disbalance'] / (x['disbalance'] + 1),
                                 acc_minus=lambda x: 1 / (x['disbalance'] + 1)).assign(acc_const=lambda x: np.maximum(x['acc_plus'], x['acc_minus'])).melt('J2', ['score', 'acc_const']),
             x='J2', y='value', hue='variable')

